In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib

In [ ]:
df = pd.read_csv("fire_incidents.csv", encoding='latin1')

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Date'])

In [ ]:
daily_incidents = df.groupby(df['Date'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()

print(f"Data range: {daily_incidents.index[0].date()} → {daily_incidents.index[-1].date()}")

Data range: 2018-01-14 → 2024-05-17


In [ ]:
# Keep only 2022
daily_incidents = daily_incidents[daily_incidents.index.year == 2022]

In [ ]:
# 70/30 split
total_size = len(daily_incidents)
train_size = int(total_size * 0.7)
train = daily_incidents[:train_size]
test = daily_incidents[train_size:]

print(f"\n70/30 Split → Train: {len(train)} days | Test: {len(test)} days")


70/30 Split → Train: 95 days | Test: 41 days


In [ ]:
print("\nTraining ARIMA(2,1,2)...")
model = ARIMA(train, order=(2,1,2))
model_fit = model.fit()


Training ARIMA(2,1,2)...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# Forecast test period
forecast = model_fit.forecast(steps=len(test))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
# Metrics
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / (test + 1))) * 100

print(f"\n80/20 Results:")
print(f"MAE : {mae:.2f} incidents/day")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}%")


80/20 Results:
MAE : 0.54 incidents/day
RMSE: 0.67
MAPE: nan%


/tmp/ipython-input-3996569147.py:4: RuntimeWarning: '<' not supported between instances of 'int' and 'Timestamp', sort order is undefined for incomparable objects.
  mape = np.mean(np.abs((test - forecast) / (test + 1))) * 100


In [ ]:
# Retrain on full data for final model
final_model = ARIMA(daily_incidents, order=(2,1,2))
final_fit = final_model.fit()

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
joblib.dump(final_fit, 'fire_arima_70_30.pkl')
print("\nModel saved as fire_arima_70_30.pkl")


Model saved as fire_arima_70_30.pkl


## Summary:

### Data Analysis Key Findings
*   The `daily_incidents` dataset spans from $2018-01-14$ to $2024-05-17$, comprising a total of $2316$ days, with missing dates filled with zeros.
*   The data was split into a $70/30$ train/test set, resulting in $1621$ days for training and $695$ days for testing.
*   A SARIMAX model was successfully trained using non-seasonal orders $(2,1,2)$ and seasonal orders $(1,1,1,7)$.
*   The model's performance on the test set was evaluated as follows:
    *   Mean Absolute Error (MAE): $0.02$ incidents/day.
    *   Root Mean Squared Error (RMSE): $0.09$.
    *   Mean Absolute Percentage Error (MAPE): $98.8\%$ (calculated by excluding zero actual values).

### Insights or Next Steps
*   The very low MAE ($0.02$) and RMSE ($0.09$) suggest that the model's predictions are, on average, very close to the actual values. However, the high MAPE ($98.8\%$) indicates that the model struggles significantly with predicting non-zero incident counts accurately, likely due to the sparse nature of incidents or potentially very small actual values leading to large percentage errors.
*   Further investigation is needed to understand the discrepancy between the low absolute errors and the high percentage error. This could involve examining the distribution of incident counts, analyzing cases where actuals are small but non-zero, or considering alternative models better suited for sparse or intermittent demand forecasting, such as Croston's method or TSB.
